# Main closed-loop study: observed future versus LSTM64

This release notebook reproduces the six main closed-loop cases used by the
manuscript figure pipeline: two fixed baselines and observed/LSTM64 MIQP cases
with and without the onsite-PV objective. The measured L14 table is deliberately
not distributed; place it under `data/private/` before execution.


## Reproducibility contract

- Inputs use repository-relative paths only.
- Thermal and PV artifacts are read from `models/`.
- The LSTM64 rollout is read from `outputs/lstm64/`.
- Simulation tables are written to `outputs/simulations/main/`.
- Gurobi is discovered through its normal Python installation and environment.


In [ ]:
import os
import sys
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "mmv4ef").is_dir() and (candidate / "models").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the MMV4EF release repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from mmv4ef.mpc_future_data import (
    apply_future_data_source,
    build_forecast_audit,
    load_lstm64_future,
)

plt.rcParams["figure.figsize"] = (12, 4)


In [ ]:
# Shared final-study implementation (no duplicated controller).
from mmv4ef.controller import *
torch.set_num_threads(1)


In [ ]:
# Shared final-study implementation (no duplicated controller).


In [ ]:
# Shared final-study implementation (no duplicated controller).


In [ ]:
# Shared final-study implementation (no duplicated controller).


## Study configuration


In [ ]:
L14_DATA_PATH = Path(os.environ.get("MMV4EF_DATA", REPO_ROOT / "data/private/l14_merged_data_with_rain.csv"))
AC_MODEL_PATH = REPO_ROOT / "models" / "thermal" / "ac_model.pth"
NV_MODEL_PATH = REPO_ROOT / "models" / "thermal" / "nv_model.pth"
PV_COEFFICIENT_PATH = REPO_ROOT / "models" / "pv" / "pv_appendix_b_best_model.csv"
LSTM64_ARTIFACT_PATH = Path(os.environ.get("MMV4EF_LSTM64_FUTURE", REPO_ROOT / "outputs/lstm64/mpc_future_disturbances.csv.gz"))
OUTPUT_DIR = Path(os.environ.get("MMV4EF_OUTPUT_DIR", REPO_ROOT / "outputs/simulations/main"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

lstm64_future = load_lstm64_future(LSTM64_ARTIFACT_PATH)
segment_split = "test+val"
test_date_policy = "union"
max_segments_raw = os.environ.get("MMV4EF_MAX_SEGMENTS", "").strip()
max_segments_to_run = int(max_segments_raw) if max_segments_raw else None
if max_segments_to_run is not None and max_segments_to_run < 1:
    raise ValueError("MMV4EF_MAX_SEGMENTS must be positive when set.")

window_mode = os.environ.get("MMV4EF_WINDOW", "full").strip().lower()
if window_mode == "smoke":
    run_window = dict(start_hhmm="07:30", end_hhmm="09:00")
    mpc_overrides = dict(horizon_min=60, mip_time_limit=600)
elif window_mode == "full":
    run_window = dict(start_hhmm="07:30", end_hhmm="19:00")
    mpc_overrides = {}
else:
    raise ValueError("MMV4EF_WINDOW must be 'full' or 'smoke'.")

pv_source_area_sqft = 5000.0
building_area_sqft = 75.0
pv_scale_factor = building_area_sqft / pv_source_area_sqft
best_model = pd.read_csv(PV_COEFFICIENT_PATH).iloc[0]
coeffs = {name: float(best_model[name]) for name in ("a1", "a2", "a3")}

SIMPLE_WEIGHTS = dict(
    energy_weight=1.667e3,
    smoothness_weight=1e-1,
    pv_self_consumption_weight=20.0,
)
FIXED_MPC_KWARGS = dict(
    T_ref_ac=27.0,
    T_nv_max=30.5,
    T_nv_soft=30.0,
    pmv_model="linear",
    rh_ac_fixed=65.0,
    rh_nv_source_col="OutdoorHumidityWindow",
    pmv_big_m=5.0,
    energy_norm_kwh=10.0,
    cop_fcu=5.0,
    cop_pfc=5.0,
    T_out_nv_ref=30.0,
    min_dwell_steps=12,
    enforce_current_rain_safety=True,
    rain_active_threshold=0.5,
    rain_lockout_min=30,
    ignore_rain_in_mpc_prediction=True,
    linearize_energy=True,
    w_terminal=2e1,
    w_switch=1e1,
    w_x_slack=1e2,
)
FCU_SUPPLY_COLS = [f"fcu{unit:02d}_supply" for unit in range(1, 6)]
PFC_SUPPLY_COLS = [f"pfc{unit:02d}_supply" for unit in range(1, 3)]




base_cfg = build_simple_pmv_cfg(**SIMPLE_WEIGHTS, use_pv_objective=False, **mpc_overrides)
pv_cfg = build_simple_pmv_cfg(**SIMPLE_WEIGHTS, use_pv_objective=True, **mpc_overrides)
pv_metric_cfg = build_simple_pmv_cfg(**SIMPLE_WEIGHTS, use_pv_objective=True, **mpc_overrides)

BASELINE_CASES = ("AC baseline", "RBC baseline")
MIQP_CASE_SPECS = (
    ("MIQP no PV | observed future", base_cfg, "no_pv", "observed"),
    ("MIQP no PV | LSTM64", base_cfg, "no_pv", "lstm64"),
    ("MIQP onsite PV | observed future", pv_cfg, "onsite_pv", "observed"),
    ("MIQP onsite PV | LSTM64", pv_cfg, "onsite_pv", "lstm64"),
)
CASE_ORDER = [*BASELINE_CASES, *(spec[0] for spec in MIQP_CASE_SPECS)]
CASE_METADATA = {
    "AC baseline": {"controller_objective": "baseline_ac", "forecast_source": "not_applicable"},
    "RBC baseline": {"controller_objective": "baseline_rbc", "forecast_source": "not_applicable"},
    **{
        name: {"controller_objective": objective, "forecast_source": source}
        for name, _, objective, source in MIQP_CASE_SPECS
    },
}

hooks = make_default_hooks(
    data_path=str(L14_DATA_PATH),
    ac_model_path=AC_MODEL_PATH,
    nv_model_path=NV_MODEL_PATH,
    test_date_policy=test_date_policy,
    segment_split=segment_split,
    min_seg_len=600,
)
segments = hooks._segments_cached()
if not segments:
    raise ValueError("No clean segments are available for the requested split.")
selected_indices = list(range(len(segments)))
if max_segments_to_run is not None:
    selected_indices = selected_indices[:max_segments_to_run]
selected_dates = [segments[index]["date"].iloc[0].date().isoformat() for index in selected_indices]

if max_segments_to_run is None and window_mode == "full":
    run_stem = "future_data_source_comparison_test_val_union_all_full"
else:
    run_count = "all" if max_segments_to_run is None else f"first{len(selected_indices)}"
    run_stem = f"future_data_source_comparison_test_val_union_{run_count}_{window_mode}"

print({
    "data_path": L14_DATA_PATH.name,
    "lstm64_path": LSTM64_ARTIFACT_PATH.name,
    "output_dir": OUTPUT_DIR.name,
    "selected_dates": selected_dates,
    "window_mode": window_mode,
    "cases": CASE_ORDER,
    "solver_available": GUROBI_OK,
})

## Fixed-policy baselines


In [ ]:
# Shared final-study implementation (no duplicated controller).


## Metrics and trajectory validation


In [ ]:
def enrich_results(
    sim_df: pd.DataFrame,
    cfg: MPCConfig,
    case_name: Optional[str] = None,
    segment_date: Optional[str] = None,
    forecast_source: Optional[str] = None,
    controller_objective: Optional[str] = None,
) -> pd.DataFrame:
    out = sim_df.copy()
    if case_name is not None:
        out['case'] = case_name
    if segment_date is not None:
        out['date'] = segment_date
    case_metadata = CASE_METADATA.get(case_name, {})
    out['forecast_source'] = (
        forecast_source
        if forecast_source is not None
        else case_metadata.get('forecast_source', 'not_applicable')
    )
    out['controller_objective'] = (
        controller_objective
        if controller_objective is not None
        else case_metadata.get('controller_objective', 'not_applicable')
    )
    out['pv_kw'] = [
        pv_available_power_kw(cfg, I_solar=i_solar, T_out=t_out)
        for i_solar, t_out in zip(out['I_solar'], out['T_out'])
    ]
    out['hvac_kw'] = [
        hvac_power_heat_exchange_kw(
            cfg,
            z_mode=row['z'],
            zone_temps=row[AC_STATE_COLS].to_numpy(dtype=float),
            fcu_supplies=row[FCU_SUPPLY_COLS].fillna(row['T_out']).to_numpy(dtype=float),
            pfc_supplies=row[PFC_SUPPLY_COLS].to_numpy(dtype=float),
            T_out=row['T_out'],
        )
        for _, row in out.iterrows()
    ]
    out['grid_kw'] = np.maximum(out['hvac_kw'] - out['pv_kw'], 0.0)
    out['export_kw'] = np.maximum(out['pv_kw'] - out['hvac_kw'], 0.0)
    out['self_kw'] = np.minimum(out['hvac_kw'], out['pv_kw'])
    return out


def sparse_progress(label: str, segment_date: str):
    def _progress(step_idx: int, total_steps: int, ts: pd.Timestamp) -> None:
        if step_idx == 0 or (step_idx + 1) % 20 == 0 or step_idx + 1 == total_steps:
            print(f'{segment_date} {label}: step {step_idx + 1}/{total_steps} at {ts}')
    return _progress


def summarize_case(df: pd.DataFrame, pv_window_threshold_kw: Optional[float] = None) -> dict:
    step_h = 1.0 / 60.0
    if pv_window_threshold_kw is None:
        pv_window_threshold_kw = max(float(df['pv_kw'].max()) * 0.05, 1e-9)
    pv_window = df['pv_kw'] > pv_window_threshold_kw

    hvac_sum = float(df['hvac_kw'].sum())
    pv_sum = float(df['pv_kw'].sum())
    self_sum = float(df['self_kw'].sum())
    switches = int((df['z'].astype(int).diff().abs().fillna(0) > 0).sum())
    window_open = 1 - df['z'].astype(int)
    zone_values = df[AC_STATE_COLS].to_numpy(dtype=float)
    below = np.maximum(22.0 - zone_values, 0.0)
    above = np.maximum(zone_values - 30.0, 0.0)
    excursion_metrics = {
        'zone_min_c': float(np.nanmin(zone_values)),
        'zone_max_c': float(np.nanmax(zone_values)),
        'zone_minutes_below_22': int(np.any(below > 0.0, axis=1).sum()),
        'zone_minutes_above_30': int(np.any(above > 0.0, axis=1).sum()),
        'max_below_22_c': float(np.nanmax(below)),
        'max_above_30_c': float(np.nanmax(above)),
    }

    return {
        'hvac_kwh': hvac_sum * step_h,
        'pv_available_kwh': pv_sum * step_h,
        'grid_kwh': float(df['grid_kw'].sum()) * step_h,
        'export_kwh': float(df['export_kw'].sum()) * step_h,
        'self_consumed_kwh': self_sum * step_h,
        'self_consumption_ratio': (self_sum / pv_sum) if pv_sum > 0 else np.nan,
        'self_sufficiency_ratio': (self_sum / hvac_sum) if hvac_sum > 0 else np.nan,
        'pv_window_hvac_kwh': float(df.loc[pv_window, 'hvac_kw'].sum()) * step_h,
        'non_pv_window_hvac_kwh': float(df.loc[~pv_window, 'hvac_kw'].sum()) * step_h,
        'pv_window_hvac_share_pct': 100.0 * float(df.loc[pv_window, 'hvac_kw'].sum()) / hvac_sum if hvac_sum > 0 else np.nan,
        'mean_temp_c': float(df['T_mean'].mean()),
        'max_temp_c': float(df['T_mean'].max()),
        'min_temp_c': float(df['T_mean'].min()),
        'ac_fraction': float(df['z'].mean()),
        'window_open_fraction': float(window_open.mean()),
        'window_open_minutes': int(window_open.sum()),
        'switch_count': switches,
        **excursion_metrics,
    }


def build_metric_table(case_results: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    max_pv = max(float(df['pv_kw'].max()) for df in case_results.values())
    pv_window_threshold_kw = max(max_pv * 0.05, 1e-9)
    metrics = pd.DataFrame.from_dict(
        {
            name: summarize_case(
                df, pv_window_threshold_kw=pv_window_threshold_kw
            )
            for name, df in case_results.items()
        },
        orient='index',
    ).loc[CASE_ORDER]

    metrics['controller_objective'] = [
        CASE_METADATA[name]['controller_objective'] for name in metrics.index
    ]
    metrics['forecast_source'] = [
        CASE_METADATA[name]['forecast_source'] for name in metrics.index
    ]

    ac_hvac = metrics.loc['AC baseline', 'hvac_kwh']
    rbc_hvac = metrics.loc['RBC baseline', 'hvac_kwh']
    min_reference_kwh = 0.05

    def savings_pct(reference_kwh: float) -> pd.Series:
        if not np.isfinite(reference_kwh) or reference_kwh <= min_reference_kwh:
            return pd.Series(np.nan, index=metrics.index, dtype=float)
        return 100.0 * (reference_kwh - metrics['hvac_kwh']) / reference_kwh

    metrics['energy_savings_vs_ac_pct'] = savings_pct(ac_hvac)
    metrics['energy_savings_vs_rbc_pct'] = savings_pct(rbc_hvac)
    metrics['energy_savings_vs_miqp_no_pv_pct'] = np.nan
    metrics['grid_reduction_vs_miqp_no_pv_kwh'] = np.nan
    metrics['load_shifted_to_pv_window_kwh'] = np.nan

    for forecast_source, no_pv_case in {
        'observed': 'MIQP no PV | observed future',
        'lstm64': 'MIQP no PV | LSTM64',
    }.items():
        reference = metrics.loc[no_pv_case]
        members = metrics['forecast_source'].eq(forecast_source)
        if reference['hvac_kwh'] > min_reference_kwh:
            metrics.loc[members, 'energy_savings_vs_miqp_no_pv_pct'] = (
                100.0
                * (
                    reference['hvac_kwh']
                    - metrics.loc[members, 'hvac_kwh']
                )
                / reference['hvac_kwh']
            )
        metrics.loc[members, 'grid_reduction_vs_miqp_no_pv_kwh'] = (
            reference['grid_kwh'] - metrics.loc[members, 'grid_kwh']
        )
        metrics.loc[members, 'load_shifted_to_pv_window_kwh'] = (
            metrics.loc[members, 'pv_window_hvac_kwh']
            - reference['pv_window_hvac_kwh']
        )
    return metrics


def metric_table_to_records(metrics: pd.DataFrame, segment_date: str) -> pd.DataFrame:
    out = metrics.reset_index().rename(columns={'index': 'case'})
    out.insert(0, 'date', segment_date)
    return out

DAILY_DISPLAY_COLS = [
    'controller_objective',
    'forecast_source',
    'hvac_kwh',
    'grid_kwh',
    'self_consumed_kwh',
    'self_consumption_ratio',
    'self_sufficiency_ratio',
    'energy_savings_vs_ac_pct',
    'energy_savings_vs_rbc_pct',
    'grid_reduction_vs_miqp_no_pv_kwh',
    'load_shifted_to_pv_window_kwh',
    'pv_window_hvac_share_pct',
    'mean_temp_c',
    'max_temp_c',
    'zone_min_c',
    'zone_max_c',
    'zone_minutes_below_22',
    'zone_minutes_above_30',
    'max_below_22_c',
    'max_above_30_c',
    'ac_fraction',
    'switch_count',
]


def _safe_path_token(value: str) -> str:
    return str(value).replace('/', '-').replace(':', '-').replace(' ', '_')


def _record_artifact(segment_date: str, artifact_type: str, path: Path) -> dict:
    return {
        'date': segment_date,
        'artifact_type': artifact_type,
        'path': str(path),
    }


def _save_figure(fig: plt.Figure, path: Path, show_figures: bool) -> None:
    fig.savefig(path, dpi=180, bbox_inches='tight')
    if show_figures:
        plt.show()
    plt.close(fig)


def plot_daily_metric_bars(metrics_df: pd.DataFrame, segment_date: str) -> plt.Figure:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    metrics_df['hvac_kwh'].plot(kind='bar', ax=axes[0, 0], title='HVAC energy')
    axes[0, 0].set_ylabel('kWh')
    metrics_df['grid_kwh'].plot(kind='bar', ax=axes[0, 1], title='Grid energy after onsite PV')
    axes[0, 1].set_ylabel('kWh')
    metrics_df[['self_consumption_ratio', 'self_sufficiency_ratio']].plot(kind='bar', ax=axes[1, 0], title='SC and SS')
    axes[1, 0].set_ylim(0, 1)
    metrics_df['load_shifted_to_pv_window_kwh'].plot(kind='bar', ax=axes[1, 1], title='Load shifted to PV window vs MIQP no PV')
    axes[1, 1].set_ylabel('kWh')
    for ax in axes.ravel():
        ax.tick_params(axis='x', rotation=30)
    fig.suptitle(f'Control case comparison metrics: {segment_date}')
    fig.tight_layout()
    return fig


def plot_daily_power_mode(case_results: Dict[str, pd.DataFrame], segment_date: str) -> plt.Figure:
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    for case_name in CASE_ORDER:
        df = case_results[case_name]
        axes[0].plot(df.index, df['hvac_kw'], label=case_name)
        axes[1].plot(df.index, df['grid_kw'], label=case_name)
        axes[2].step(df.index, df['z'], where='post', label=case_name)
    axes[0].plot(
        case_results[CASE_ORDER[0]].index,
        case_results[CASE_ORDER[0]]['pv_kw'],
        color='black',
        linestyle='--',
        label='PV available',
    )
    axes[0].set_ylabel('HVAC / PV kW')
    axes[1].set_ylabel('Grid kW')
    axes[2].set_ylabel('Mode z')
    axes[2].set_yticks([0, 1], ['NV/CC', 'AC'])
    for ax in axes:
        ax.legend(loc='best')
    fig.suptitle(f'Control case comparison: {segment_date} ({window_mode})')
    fig.tight_layout()
    return fig


def plot_daily_thermal_trajectory(case_results: Dict[str, pd.DataFrame], segment_date: str) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(14, 5))
    for case_name in CASE_ORDER:
        df = case_results[case_name]
        ax.plot(df.index, df['T_mean'], label=case_name)
    ax.axhline(AC_ZONE_SETPOINT_C, color='black', linestyle='--', linewidth=1.0, label='27 C setpoint')
    ax.set_ylabel('Mean zone temperature (C)')
    ax.set_title(f'Thermal trajectory: {segment_date}')
    ax.legend(loc='best')
    fig.tight_layout()
    return fig


def save_daily_outputs(
    case_results: Dict[str, pd.DataFrame],
    daily_metrics: pd.DataFrame,
    segment_date: str,
    output_dir: Path,
    run_stem: str,
    show_figures: bool = False,
) -> List[dict]:
    day_token = _safe_path_token(segment_date)
    day_dir = output_dir / 'per_day' / day_token
    day_dir.mkdir(parents=True, exist_ok=True)
    artifact_records = []

    summary_csv = day_dir / f'{run_stem}_{day_token}_summary.csv'
    table_csv = day_dir / f'{run_stem}_{day_token}_display_table.csv'
    daily_metrics.to_csv(summary_csv)
    daily_metrics[DAILY_DISPLAY_COLS].round(3).to_csv(table_csv)
    artifact_records.append(_record_artifact(segment_date, 'summary_csv', summary_csv))
    artifact_records.append(_record_artifact(segment_date, 'display_table_csv', table_csv))

    metrics_png = day_dir / f'{run_stem}_{day_token}_metrics_plot.png'
    _save_figure(plot_daily_metric_bars(daily_metrics, segment_date), metrics_png, show_figures)
    artifact_records.append(_record_artifact(segment_date, 'metrics_plot_png', metrics_png))

    power_mode_png = day_dir / f'{run_stem}_{day_token}_power_mode.png'
    _save_figure(plot_daily_power_mode(case_results, segment_date), power_mode_png, show_figures)
    artifact_records.append(_record_artifact(segment_date, 'power_mode_png', power_mode_png))

    thermal_png = day_dir / f'{run_stem}_{day_token}_thermal_trajectory.png'
    _save_figure(plot_daily_thermal_trajectory(case_results, segment_date), thermal_png, show_figures)
    artifact_records.append(_record_artifact(segment_date, 'thermal_trajectory_png', thermal_png))

    return artifact_records


PAIR_METRICS = [
    'hvac_kwh', 'grid_kwh', 'export_kwh', 'self_consumed_kwh',
    'self_consumption_ratio', 'self_sufficiency_ratio',
    'pv_window_hvac_kwh', 'mean_temp_c', 'max_temp_c', 'min_temp_c',
    'zone_min_c', 'zone_max_c', 'zone_minutes_below_22',
    'zone_minutes_above_30', 'max_below_22_c', 'max_above_30_c',
    'ac_fraction', 'window_open_fraction', 'window_open_minutes',
    'switch_count',
]


def validate_case_timeseries(
    df: pd.DataFrame,
    case_name: str,
    segment_date: str,
    expected_index: pd.DatetimeIndex,
) -> dict:
    required = ['z', 'T_mean', 'hvac_kw', 'grid_kw', *AC_STATE_COLS]
    is_miqp = case_name.startswith('MIQP ')
    if is_miqp:
        required.append('rain_force_ac')
    missing_columns = sorted(set(required) - set(df.columns))
    required_missing_values = (
        int(df.loc[:, [c for c in required if c in df]].isna().sum().sum())
        + len(missing_columns) * len(df)
    )
    z_mode = df['z'].astype(int)
    dwell_minutes = z_mode.groupby(z_mode.ne(z_mode.shift()).cumsum()).size()
    if is_miqp and 'rain_force_ac' in df:
        rain_force_active = df['rain_force_ac'].astype(bool)
        rain_active_minutes = int(rain_force_active.sum())
        rain_lockout_violations = int(
            (rain_force_active & z_mode.ne(1)).sum()
        )
        rain_validation_scope = 'current observed rain plus lockout'
    else:
        rain_active_minutes = 0
        rain_lockout_violations = 0
        rain_validation_scope = 'not_applicable'
    return {
        'date': segment_date,
        'case': case_name,
        'expected_rows': len(expected_index),
        'actual_rows': len(df),
        'exact_index': bool(df.index.equals(expected_index)),
        'missing_required_columns': '|'.join(missing_columns),
        'required_missing_values': required_missing_values,
        'rain_validation_scope': rain_validation_scope,
        'rain_lockout_active_minutes': rain_active_minutes,
        'rain_lockout_violations': rain_lockout_violations,
        'minimum_realized_dwell_min': (
            int(dwell_minutes.min()) if len(dwell_minutes) else 0
        ),
    }


## Run the selected dates and write source tables


In [ ]:
if not GUROBI_OK:
    raise RuntimeError("Gurobi is not available; MIQP cases cannot run.")

all_case_results = {case: [] for case in CASE_ORDER}
daily_summary_records = []
timeseries_records = []
forecast_audit_records = []
validation_records = []
error_records = []

for run_position, local_index in enumerate(selected_indices, start=1):
    segment_date = segments[local_index]["date"].iloc[0].date().isoformat()
    print(f"Running segment {run_position}/{len(selected_indices)}: {segment_date}")
    try:
        case_results = {}
        ac_frame = simulate_policy_day(
            "AC baseline", hooks, pv_metric_cfg, day_idx=local_index,
            policy="ac", **run_window,
        )
        case_results["AC baseline"] = enrich_results(
            ac_frame, pv_metric_cfg, "AC baseline", segment_date,
        )
        rbc_frame = simulate_policy_day(
            "RBC baseline", hooks, pv_metric_cfg, day_idx=local_index,
            policy="rbc", **run_window,
        )
        case_results["RBC baseline"] = enrich_results(
            rbc_frame, pv_metric_cfg, "RBC baseline", segment_date,
        )

        for case_name, case_cfg, objective, forecast_source in MIQP_CASE_SPECS:
            hooks.reset_cache()
            result = simulate_day(
                day_idx=local_index,
                hooks=hooks,
                cfg=case_cfg,
                init_state_cols=hooks.state_cols,
                progress_fn=sparse_progress(case_name, segment_date),
                forecast_source=forecast_source,
                lstm64_future=lstm64_future if forecast_source == "lstm64" else None,
                forecast_case_name=case_name,
                forecast_controller_objective=objective,
                forecast_audit_records=forecast_audit_records,
                **run_window,
            )
            case_results[case_name] = enrich_results(
                result,
                pv_metric_cfg,
                case_name,
                segment_date,
                forecast_source=forecast_source,
                controller_objective=objective,
            )

        expected_index = case_results["AC baseline"].index
        for case_name, frame in case_results.items():
            validation_records.append(
                validate_case_timeseries(frame, case_name, segment_date, expected_index)
            )
            all_case_results[case_name].append(frame)
            timeseries_records.append(frame.reset_index())
        daily_summary_records.append(
            metric_table_to_records(build_metric_table(case_results), segment_date)
        )
    except Exception as exc:
        error_records.append({
            "date": segment_date,
            "segment_index": local_index,
            "error_type": type(exc).__name__,
            "error": repr(exc),
        })
        print(f"ERROR on {segment_date}: {exc!r}")

if not timeseries_records:
    raise RuntimeError("No segments completed successfully.")
missing_cases = [case for case in CASE_ORDER if not all_case_results[case]]
if missing_cases:
    raise RuntimeError(f"Missing completed results for cases: {missing_cases}")

aggregate_case_results = {
    name: pd.concat(frames, axis=0).sort_index()
    for name, frames in all_case_results.items()
}
aggregate_summary_df = (
    build_metric_table(aggregate_case_results)
    .reset_index()
    .rename(columns={"index": "case"})
)
daily_summary_df = pd.concat(daily_summary_records, ignore_index=True)
all_timeseries_df = pd.concat(timeseries_records, ignore_index=True)
forecast_audit_df = pd.concat(forecast_audit_records, ignore_index=True)
validation_df = pd.DataFrame(validation_records)
errors_df = pd.DataFrame(
    error_records,
    columns=["date", "segment_index", "error_type", "error"],
)

paths = {
    "aggregate": OUTPUT_DIR / f"{run_stem}_aggregate_summary.csv",
    "daily": OUTPUT_DIR / f"{run_stem}_daily_summary.csv",
    "timeseries": OUTPUT_DIR / f"{run_stem}_timeseries.csv",
    "forecast_audit": OUTPUT_DIR / f"{run_stem}_forecast_audit.csv.gz",
    "validation": OUTPUT_DIR / f"{run_stem}_validation.csv",
    "errors": OUTPUT_DIR / f"{run_stem}_errors.csv",
    "metadata": OUTPUT_DIR / f"{run_stem}_run_metadata.json",
}
aggregate_summary_df.to_csv(paths["aggregate"], index=False)
daily_summary_df.to_csv(paths["daily"], index=False)
all_timeseries_df.to_csv(paths["timeseries"], index=False)
forecast_audit_df.to_csv(paths["forecast_audit"], index=False, compression="gzip")
validation_df.to_csv(paths["validation"], index=False)
errors_df.to_csv(paths["errors"], index=False)

validation_failures = []
if not errors_df.empty:
    validation_failures.append(f"{len(errors_df)} segment errors")
invalid = validation_df.loc[
    ~validation_df["exact_index"]
    | validation_df["required_missing_values"].ne(0)
    | validation_df["rain_lockout_violations"].ne(0)
]
if not invalid.empty:
    validation_failures.append(f"{len(invalid)} invalid case trajectories")
if set(forecast_audit_df["forecast_source"].unique()) != {"observed", "lstm64"}:
    validation_failures.append("forecast source contract mismatch")
if forecast_audit_df.duplicated(["case", "decision_time", "horizon_step"]).any():
    validation_failures.append("duplicate forecast-audit keys")
if not forecast_audit_df["rain_status"].eq(0.0).all():
    validation_failures.append("future rain was not neutralized")

start = pd.Timestamp(f"2000-01-01 {run_window['start_hhmm']}")
end = pd.Timestamp(f"2000-01-01 {run_window['end_hhmm']}")
minutes_per_day = int((end - start).total_seconds() / 60)
completed_dates = daily_summary_df["date"].nunique()
expected_daily_rows = completed_dates * len(CASE_ORDER)
expected_timeseries_rows = expected_daily_rows * minutes_per_day
expected_audit_rows = (
    completed_dates
    * len(MIQP_CASE_SPECS)
    * (minutes_per_day // base_cfg.ctrl_period_min)
    * (base_cfg.horizon_min // base_cfg.ctrl_period_min)
)
for label, actual, expected in (
    ("daily rows", len(daily_summary_df), expected_daily_rows),
    ("timeseries rows", len(all_timeseries_df), expected_timeseries_rows),
    ("forecast audit rows", len(forecast_audit_df), expected_audit_rows),
):
    if actual != expected:
        validation_failures.append(f"{label}: expected {expected}, observed {actual}")
if max_segments_to_run is None and window_mode == "full" and completed_dates != 22:
    validation_failures.append(f"full study requires 22 dates; observed {completed_dates}")

run_metadata = {
    "study": "Observed and LSTM64 future disturbances in MPC",
    "input_paths": {
        "observed_data": L14_DATA_PATH.name,
        "lstm64_future": LSTM64_ARTIFACT_PATH.name,
        "ac_model": str(AC_MODEL_PATH.relative_to(REPO_ROOT)),
        "nv_model": str(NV_MODEL_PATH.relative_to(REPO_ROOT)),
        "pv_coefficients": str(PV_COEFFICIENT_PATH.relative_to(REPO_ROOT)),
    },
    "selected_dates": selected_dates,
    "window_mode": window_mode,
    "cases": CASE_ORDER,
    "row_counts": {
        "daily": len(daily_summary_df),
        "timeseries": len(all_timeseries_df),
        "forecast_audit": len(forecast_audit_df),
    },
    "validation_passed": not validation_failures,
}
paths["metadata"].write_text(
    json.dumps(run_metadata, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print({
    "completed_dates": completed_dates,
    "output_dir": OUTPUT_DIR.name,
    "timeseries": paths["timeseries"].name,
    "validation_failures": validation_failures,
})
if validation_failures:
    raise RuntimeError("; ".join(validation_failures))
aggregate_summary_df.round(3)


## Figure generation

After the full run succeeds, execute `python scripts/reproduce_figures.py` from
the repository root. That command consumes the validated tables written above
and creates the manuscript figures.
